In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EvalPrediction,
)
from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
)
import matplotlib.pyplot as plt
import os

In [ ]:
#Load (cleaned) datasets
train_df = pd.read_csv("../datasets/clean/train.csv")
val_df = pd.read_csv("../datasets/clean/val.csv")
test_df = pd.read_csv("../datasets/clean/test.csv")

print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

In [ ]:
# Convert to DatasetDict (for HF)
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})

In [ ]:
#Tokenize model
MODEL_NAME = "hannybal/disaster-twitter-xlm-roberta-al"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#Preprocess data
def preprocess(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

encoded_dataset = dataset.map(preprocess, batched=True)

In [ ]:
#Load model
model = AutoModelForSequenceClassification.from_pretrained(
    "hannybal/disaster-twitter-xlm-roberta-al",  
    num_labels=2
)

In [ ]:
#Define metrics function
def compute_metrics(p: EvalPrediction):
    y_pred = np.argmax(p.predictions, axis=1)
    y_true = p.label_ids
    y_probs = torch.nn.functional.softmax(torch.tensor(p.predictions), dim=1)[:, 1].numpy()
    
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "log_loss": log_loss(y_true, y_probs)
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="../models/finetuned-model",
    num_train_epochs=5,  
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="../logs/hf_trainer",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to="none",  #disable wandb
)

In [ ]:
#Setup trainer
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
#Finetune model
train_result = trainer.train()
trainer.save_model("../models/finetuned-model")

In [ ]:
log_history = trainer.state.log_history
losses = [log["loss"] for log in log_history if "loss" in log]
steps = [log["step"] for log in log_history if "loss" in log]

#Plot loss curve
plt.plot(steps, losses)
plt.title("Training Loss Over Time")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#Evaluate on test data
test_results = trainer.evaluate(encoded_dataset["test"])
print("Final Evaluation on Test Set:")
for k, v in test_results.items():
    print(f"{k}: {v:.4f}")

In [ ]:
#Save metrics
pd.DataFrame([test_results]).to_csv("../notebooks/logs/test_metrics.csv", index=False)
print("Saved test metrics to logs/test_metrics.csv")

# Save tokenizer + weights
tokenizer.save_pretrained("../models/finetuned-model")
model.save_pretrained("../models/finetuned-model")